In [156]:
import pandas as pd

df = pd.read_csv("cc_calls.csv")   # change file name if needed

In [157]:
df_safe=df.copy()

In [158]:
print(df.columns)

Index(['Contact_ID', 'Call_Date', 'Direction', 'cc_care_package',
       'cc_care_package_discussed', 'cc_urgency_getting_on_site',
       'cc_external_consultant', 'cc_agent_cross_sell_attempt',
       'cc_customer_issues_concerns',
       'cc_business_struggles_financial_hardship', 'cc_call_initiated_by',
       'cc_questionnaire_completion', 'cc_chasing_response',
       'cc_issues_within_questionnaire', 'cc_login_issues',
       'cc_platform_issues', 'cc_dissatisfaction_time_to_complete',
       'cc_process_complexity_concerns', 'cc_questions_harder_than_expected',
       'cc_dissatisfaction_support', 'cc_contractor_sentiment',
       'cc_contractor_sentiment_start_score',
       'cc_contractor_sentiment_end_score',
       'cc_contractor_sentiment_overall_score',
       'cc_contractor_sentiment_issues_score', 'cc_pricing_mentioned',
       'cc_pricing_sentiment_impact', 'cc_refund_discussed',
       'cc_contractor_suggest_leave', 'cc_contractor_complained', 'Co_Ref',
       'Analys

In [159]:
df['Call_Date'] = pd.to_datetime(df['Call_Date'], dayfirst=True, errors='coerce')

In [160]:
df['Call_Date'].isnull().sum()

np.int64(19242)

Create new features for call day,month and year

In [161]:
df['Call_Year'] = df['Call_Date'].dt.year
df['Call_Month'] = df['Call_Date'].dt.month
df['Call_Day'] = df['Call_Date'].dt.day

In [162]:
print(type(df['Co_Ref'].iloc[0]))

<class 'str'>


In [163]:
print("Total columns:", len(df.columns))

Total columns: 35


In [164]:
print(df.dtypes)

Contact_ID                                         float64
Call_Date                                   datetime64[us]
Direction                                              str
cc_care_package                                        str
cc_care_package_discussed                              str
cc_urgency_getting_on_site                             str
cc_external_consultant                                 str
cc_agent_cross_sell_attempt                            str
cc_customer_issues_concerns                            str
cc_business_struggles_financial_hardship               str
cc_call_initiated_by                                   str
cc_questionnaire_completion                            str
cc_chasing_response                                    str
cc_issues_within_questionnaire                         str
cc_login_issues                                        str
cc_platform_issues                                     str
cc_dissatisfaction_time_to_complete                    s

In [165]:
for col in df.columns:
    print("\nColumn Name:", col)
    print("Unique Values:", df[col].unique())


Column Name: Contact_ID
Unique Values: [6.25513e+11 5.91087e+11 5.65091e+11 5.93975e+11 6.22282e+11 6.80316e+11
 6.79955e+11 5.92748e+11 6.52191e+11 6.88425e+11 6.80941e+11 6.25851e+11
 6.89008e+11 6.88424e+11 5.96363e+11 6.89199e+11 6.89552e+11 6.84557e+11
 6.84954e+11 5.94137e+11 6.84558e+11 6.77234e+11 6.77090e+11 5.92600e+11
 5.95746e+11 6.90632e+11 6.78084e+11 6.87553e+11 5.90559e+11 6.24978e+11
 6.87862e+11 6.85920e+11 5.94293e+11 5.95603e+11 6.89718e+11 6.22846e+11
 6.76349e+11 6.78400e+11 5.94703e+11 5.90914e+11 5.91754e+11 6.22409e+11
 5.93696e+11 5.95301e+11 5.92601e+11 6.88585e+11 6.85919e+11 5.96041e+11
 6.83926e+11 6.89009e+11 6.88889e+11 6.22283e+11 5.91235e+11 5.91634e+11
 6.26392e+11 6.88015e+11 6.84787e+11 6.85631e+11 5.95465e+11 6.25852e+11
 6.87552e+11 5.65228e+11 6.86118e+11 6.89200e+11 5.91917e+11 6.85446e+11
 5.95302e+11 6.76350e+11 6.87353e+11 6.50532e+11 6.78698e+11 6.91673e+11
 6.24343e+11 6.87352e+11 6.84210e+11 6.77411e+11 6.89717e+11 6.77235e+11
 5.97368e+1

In [166]:
unnamed_cols = [col for col in df.columns if "Unnamed" in col]
print("Unnamed Columns:", unnamed_cols)
print(len(unnamed_cols))

Unnamed Columns: []
0


In [167]:
print(df['Direction'].value_counts(dropna=False))

Direction
OUT_BOUND    24871
IN_BOUND      8011
Name: count, dtype: int64


Converting the dirty cc_care_package column to the actual package which was discussed in the call

In [168]:
import numpy as np
import re

def clean_package(val):
    if pd.isna(val):
        return 'Unknown'
    
    val = str(val).lower()
    
    # Remove template junk
    if '[' in val:
        return 'Unknown'
    
    # Priority order (final decision)
    if 'premier' in val or 'premium' in val:
        return 'Premier'
    
    if 'express' in val or 'fast' in val:
        return 'Express'
    
    if 'standard' in val:
        return 'Standard'
    
    if 'assist' in val:
        return 'Assisted'
    
    if 'not discussed' in val:
        return 'Not Discussed'
    
    return 'Unknown'


df['cc_care_package_clean'] = df['cc_care_package'].apply(clean_package)

In [169]:
df['cc_care_package_clean'].unique()

<StringArray>
['Standard', 'Premier', 'Express', 'Assisted', 'Not Discussed', 'Unknown']
Length: 6, dtype: str

In [170]:
df['cc_care_package_discussed'] = df['cc_care_package_discussed'].replace(
    ['[Yes/No]'], 'Unknown'
)

In [171]:
df['cc_care_package_discussed'].unique()

<StringArray>
['Yes', 'No', nan, 'Unknown']
Length: 4, dtype: str